# Rust LogisticRegression vs sklearn — grid benchmark

Sweeps a grid of dataset shapes and records, per shape:

- **fit time** and **predict time** for both estimators (best-of N repeats)
- **accuracy** on a held-out split
- **agreement** between the two models' predictions
- **relative L2 difference** of the learned coefficients

Both estimators use `fit_intercept=False` (our adapter requires it) so the
comparison is apples-to-apples. We sweep rows and cols **gradually** so it is
easy to see at which shape (if any) behaviour diverges.

In [1]:
import sys, os, time
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as SklearnLR

sys.path.insert(0, os.path.abspath(os.path.join('..', 'adapters')))
from LogisticRegression import LogisticRegression as RustLR

REPEATS = 3       # best-of timing
C = 1.0
MAX_ITER = 100
TOL = 1e-4

ROWS = [10, 100, 1000, 10000, 50000, 100000, 500000]
COLS = [10, 50, 100, 200]
print('grid:', len(ROWS) * len(COLS), 'shapes')

grid: 28 shapes


In [2]:
def timeit(fn, repeats=REPEATS):
    best, out = float('inf'), None
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = fn()
        best = min(best, time.perf_counter() - t0)
    return best, out

def safe_div(a, b):
    return a / b if b else float('nan')

def make_data(n, d, seed=0):
    n_informative = min(max(2, d // 2), d)
    X, y = make_classification(n_samples=n, n_features=d,
                               n_informative=n_informative, random_state=seed)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    Xtr = np.ascontiguousarray(Xtr, dtype=np.float32)
    Xte = np.ascontiguousarray(Xte, dtype=np.float32)
    ytr = ytr.astype(np.float32)
    return Xtr, Xte, ytr, yte

def bench_shape(n, d):
    Xtr, Xte, ytr, yte = make_data(n, d)
    row = {'rows': n, 'cols': d}
    try:
        sk = SklearnLR(C=C, fit_intercept=False, max_iter=MAX_ITER, tol=TOL, solver='lbfgs')
        row['sk_fit'], _ = timeit(lambda: sk.fit(Xtr, ytr))
        row['sk_pred'], sk_p = timeit(lambda: sk.predict(Xte))
        row['sk_acc'] = (sk_p == yte).mean()

        rs = RustLR(C=C, fit_intercept=False, max_iter=MAX_ITER, tol=TOL, solver='lbfgs')
        row['rs_fit'], _ = timeit(lambda: rs.fit(Xtr, ytr))
        row['rs_pred'], rs_p = timeit(lambda: rs.predict(Xte))
        row['rs_acc'] = (rs_p == yte).mean()

        row['fit_speedup'] = safe_div(row['sk_fit'], row['rs_fit'])
        row['pred_speedup'] = safe_div(row['sk_pred'], row['rs_pred'])
        row['agreement'] = (rs_p == sk_p).mean()
        skc, rsc = np.ravel(sk.coef_), np.ravel(rs.coef_)
        row['coef_reldiff'] = np.linalg.norm(rsc - skc) / (np.linalg.norm(skc) + 1e-12)
        row['rs_coef_norm'] = float(np.linalg.norm(rsc))
        row['status'] = 'ok'
    except Exception as e:
        row['status'] = f'ERROR: {type(e).__name__}: {e}'
    return row

In [3]:
results = []
for n in ROWS:
    for d in COLS:
        r = bench_shape(n, d)
        results.append(r)
        if r['status'] == 'ok':
            print(f"n={n:>7} d={d:>3} | fit {r['fit_speedup']:6.2f}x  acc ours={r['rs_acc']:.3f} sk={r['sk_acc']:.3f}  "
                  f"agree={r['agreement']:.3f}  coef_reldiff={r['coef_reldiff']:.2e}  rs_coef_norm={r['rs_coef_norm']:.2e}")
        else:
            print(f"n={n:>7} d={d:>3} | {r['status']}")
df = pd.DataFrame(results)
print('\ndone')

n=     10 d= 10 | fit   2.04x  acc ours=1.000 sk=1.000  agree=1.000  coef_reldiff=2.64e-01  rs_coef_norm=9.16e-01
n=     10 d= 50 | fit  23.46x  acc ours=0.500 sk=1.000  agree=0.500  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=     10 d=100 | fit  12.81x  acc ours=1.000 sk=0.500  agree=0.500  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=     10 d=200 | fit  19.65x  acc ours=1.000 sk=1.000  agree=1.000  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=    100 d= 10 | fit   1.51x  acc ours=0.800 sk=0.800  agree=1.000  coef_reldiff=1.14e-01  rs_coef_norm=1.46e+00
n=    100 d= 50 | fit  24.22x  acc ours=0.500 sk=0.850  agree=0.450  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=    100 d=100 | fit  27.26x  acc ours=0.650 sk=0.650  agree=0.400  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=    100 d=200 | fit  29.11x  acc ours=0.450 sk=0.700  agree=0.450  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=   1000 d= 10 | fit   0.73x  acc ours=0.915 sk=0.905  agree=0.990  coef_reldiff=2.96e-

/home/bl4ck_r4bbit/Schreibtisch/TUBerlin/MLMMI-Project/rust_experiments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/bl4ck_r4bbit/Schreibtisch/TUBerlin/MLMMI-Project/rust_experiments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).


n=   1000 d=200 | fit  26.81x  acc ours=0.530 sk=0.750  agree=0.490  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=  10000 d= 10 | fit   0.92x  acc ours=0.857 sk=0.857  agree=1.000  coef_reldiff=7.31e-04  rs_coef_norm=1.09e+00
n=  10000 d= 50 | fit  17.38x  acc ours=0.499 sk=0.802  agree=0.476  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=  10000 d=100 | fit 203.93x  acc ours=0.508 sk=0.832  agree=0.463  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=  10000 d=200 | fit 208.35x  acc ours=0.497 sk=0.827  agree=0.487  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=  50000 d= 10 | fit   1.23x  acc ours=0.811 sk=0.811  agree=1.000  coef_reldiff=3.99e-04  rs_coef_norm=1.24e+00
n=  50000 d= 50 | fit  48.70x  acc ours=0.503 sk=0.822  agree=0.499  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=  50000 d=100 | fit  51.03x  acc ours=0.490 sk=0.840  agree=0.477  coef_reldiff=1.00e+00  rs_coef_norm=0.00e+00
n=  50000 d=200 | fit  56.31x  acc ours=0.500 sk=0.828  agree=0.494  coef_reldiff=1.00e+

In [4]:
pd.set_option('display.float_format', lambda v: f'{v:.4g}')
cols = ['rows','cols','sk_fit','rs_fit','fit_speedup','sk_acc','rs_acc','agreement','coef_reldiff','rs_coef_norm','status']
df[cols]

,rows,cols,sk_fit,rs_fit,fit_speedup,sk_acc,rs_acc,agreement,coef_reldiff,rs_coef_norm,status
0,10,10,0.004162,0.002044,2.036,1,1,1,0.2644,0.9163,ok
1,10,50,0.005781,0.0002464,23.46,1,0.5,0.5,1,0,ok
2,10,100,0.007078,0.0005525,12.81,0.5,1,0.5,1,0,ok
3,10,200,0.008946,0.0004553,19.65,1,1,1,1,0,ok
4,100,10,0.00588,0.003894,1.51,0.8,0.8,1,0.1139,1.458,ok
5,100,50,0.01689,0.0006974,24.22,0.85,0.5,0.45,1,0,ok
6,100,100,0.01812,0.0006647,27.26,0.65,0.65,0.4,1,0,ok
7,100,200,0.01519,0.0005218,29.11,0.7,0.45,0.45,1,0,ok
8,1000,10,0.008123,0.01113,0.7297,0.905,0.915,0.99,0.02956,2.378,ok
9,1000,50,0.01484,0.001966,7.548,0.79,0.44,0.46,1,0,ok


### Correctness flag

A shape where our model failed to optimise shows up as `rs_coef_norm ≈ 0`,
`coef_reldiff ≈ 1.0`, `agreement ≈ 0.5`, and `rs_acc ≈ 0.5` (random). The cell
below lists any such shapes.

In [5]:
ok = df[df['status'] == 'ok'].copy()
bad = ok[ok['rs_coef_norm'] < 1e-6]
if len(bad):
    print('Shapes where ours returned ~zero coefficients (optimiser made no progress):')
    display(bad[['rows','cols','rs_acc','agreement','coef_reldiff']])
else:
    print('No degenerate (all-zero) coefficient cases.')

Shapes where ours returned ~zero coefficients (optimiser made no progress):


,rows,cols,rs_acc,agreement,coef_reldiff
1,10,50,0.5,0.5,1
2,10,100,1,0.5,1
3,10,200,1,1,1
5,100,50,0.5,0.45,1
6,100,100,0.65,0.4,1
7,100,200,0.45,0.45,1
9,1000,50,0.44,0.46,1
10,1000,100,0.54,0.475,1
11,1000,200,0.53,0.49,1
13,10000,50,0.4995,0.476,1
